## Problem Statement : Hospital Patient Data Analysis

Context: A hospital maintains patient records including admission details, department, diagnosis, doctor, and bill amount. You have two datasets: one with patient info and another with billing details. Some patients have blank bill amounts, and there are multiple rows for the same patient due to follow-ups.

In [1]:
import pandas as pd
import numpy as np

#### 1. Load the patient dataset and show summary with info().

In [2]:
# Load patient dataset
patients = pd.read_csv('Patient_Data.csv')

# Display first five rows
print(patients.head())

   PatientID     Name   Department     Doctor  BillAmount  ReceptionistID  \
0        101    Alice   Cardiology  Dr. Smith      5000.0               1   
1        102      Bob    Neurology   Dr. John         NaN               2   
2        103  Charlie  Orthopedics    Dr. Lee      7500.0               1   
3        104    David   Cardiology  Dr. Smith      6200.0               3   
4        105      Eva  Dermatology   Dr. Rose         NaN               2   

        CheckInTime  
0  2023-01-10 09:00  
1  2023-01-11 10:30  
2  2023-01-12 11:00  
3  2023-01-13 12:00  
4  2023-01-14 08:45  


In [3]:
# Patients information

patients.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


#### 2. Select only the columns relevant for billing:
#### ['PatientID', 'Department', 'Doctor', 'BillAmount'].

In [4]:
billing_info = patients[['PatientID', 'Department', 'Doctor', 'BillAmount']]
print(billing_info.head())

   PatientID   Department     Doctor  BillAmount
0        101   Cardiology  Dr. Smith      5000.0
1        102    Neurology   Dr. John         NaN
2        103  Orthopedics    Dr. Lee      7500.0
3        104   Cardiology  Dr. Smith      6200.0
4        105  Dermatology   Dr. Rose         NaN


#### 3. Drop administrative columns like 
#### ['ReceptionistID', 'CheckInTime'].

In [5]:
patients = patients.drop(columns=['ReceptionistID','CheckInTime'])
print(patients.head())

   PatientID     Name   Department     Doctor  BillAmount
0        101    Alice   Cardiology  Dr. Smith      5000.0
1        102      Bob    Neurology   Dr. John         NaN
2        103  Charlie  Orthopedics    Dr. Lee      7500.0
3        104    David   Cardiology  Dr. Smith      6200.0
4        105      Eva  Dermatology   Dr. Rose         NaN


#### 4. Use groupby to find total bill amount per department.

In [6]:
department_bill = patients.groupby('Department')["BillAmount"].sum()
print(department_bill)

Department
Cardiology     16200.0
Dermatology        0.0
Neurology          0.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64


#### 5. Remove duplicate patient records based on PatientID.

In [7]:
patients = patients.drop_duplicates(subset='PatientID')
print(patients.head())

   PatientID     Name   Department     Doctor  BillAmount
0        101    Alice   Cardiology  Dr. Smith      5000.0
1        102      Bob    Neurology   Dr. John         NaN
2        103  Charlie  Orthopedics    Dr. Lee      7500.0
3        104    David   Cardiology  Dr. Smith      6200.0
4        105      Eva  Dermatology   Dr. Rose         NaN


#### 6. Fill missing BillAmount values with the mean bill amount.

In [8]:
mean_bill = patients["BillAmount"].mean()
print("Mean Bill Amount:", mean_bill)

patients["BillAmount"] = patients["BillAmount"].fillna(mean_bill)

print(patients)

Mean Bill Amount: 6233.333333333333
   PatientID     Name   Department     Doctor   BillAmount
0        101    Alice   Cardiology  Dr. Smith  5000.000000
1        102      Bob    Neurology   Dr. John  6233.333333
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000
3        104    David   Cardiology  Dr. Smith  6200.000000
4        105      Eva  Dermatology   Dr. Rose  6233.333333


#### 7. Merge the billing dataset with patient dataset on PatientID.

In [9]:
billing = pd.read_csv("Billing_Data.csv")
print(billing.head())

   PatientID  InsuranceCovered  FinalAmount
0        101              2000         3000
1        102              1500         3500
2        103              2500         5000
3        104              3000         3200
4        105              1000         4000


In [10]:
merged_data = pd.merge(
    patients,
    billing,
    on = "PatientID",
    how = "inner"
)
print(merged_data.head())

   PatientID     Name   Department     Doctor   BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith  5000.000000              2000   
1        102      Bob    Neurology   Dr. John  6233.333333              1500   
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000              2500   
3        104    David   Cardiology  Dr. Smith  6200.000000              3000   
4        105      Eva  Dermatology   Dr. Rose  6233.333333              1000   

   FinalAmount  
0         3000  
1         3500  
2         5000  
3         3200  
4         4000  


#### 8. Concatenate an additional DataFrame that contains new patients for the current week (row-wise).

In [11]:
new_patients = pd.DataFrame({
    "PatientID":[205, 210],
    "Department":["Cardiology", "Neurology"],
    "Doctor":["Dr. Raj","Dr. Lee"],
    "BillAmount":[15000, 18000]
})
updated = pd.concat([merged_data, new_patients])
print(updated.tail())

   PatientID     Name   Department     Doctor    BillAmount  InsuranceCovered  \
2        103  Charlie  Orthopedics    Dr. Lee   7500.000000            2500.0   
3        104    David   Cardiology  Dr. Smith   6200.000000            3000.0   
4        105      Eva  Dermatology   Dr. Rose   6233.333333            1000.0   
0        205      NaN   Cardiology    Dr. Raj  15000.000000               NaN   
1        210      NaN    Neurology    Dr. Lee  18000.000000               NaN   

   FinalAmount  
2       5000.0  
3       3200.0  
4       4000.0  
0          NaN  
1          NaN  


#### 9. Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).

In [12]:
new_column = pd.DataFrame({
    "InsuranceCovered": [12000, 15000],
    "FinalAmount": [3000, 3000]
})



updated = updated.reset_index(drop=True)
new_column = new_column.reset_index(drop=True)

updated = updated.drop(
    columns=["InsuranceCovered", "FinalAmount"],
    errors="ignore"
)

final_dataset = pd.concat([updated, new_column], axis=1)

print(final_dataset.head())
print(final_dataset.info())

   PatientID     Name   Department     Doctor   BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith  5000.000000           12000.0   
1        102      Bob    Neurology   Dr. John  6233.333333           15000.0   
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000               NaN   
3        104    David   Cardiology  Dr. Smith  6200.000000               NaN   
4        105      Eva  Dermatology   Dr. Rose  6233.333333               NaN   

   FinalAmount  
0       3000.0  
1       3000.0  
2          NaN  
3          NaN  
4          NaN  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   PatientID         7 non-null      int64  
 1   Name              5 non-null      object 
 2   Department        7 non-null      object 
 3   Doctor            7 non-null      object 
 4   BillAmount        7 non-null  

In [13]:
print(final_dataset.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   PatientID         7 non-null      int64  
 1   Name              5 non-null      object 
 2   Department        7 non-null      object 
 3   Doctor            7 non-null      object 
 4   BillAmount        7 non-null      float64
 5   InsuranceCovered  2 non-null      float64
 6   FinalAmount       2 non-null      float64
dtypes: float64(3), int64(1), object(3)
memory usage: 524.0+ bytes
None


In [14]:
print(final_dataset.head())

   PatientID     Name   Department     Doctor   BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith  5000.000000           12000.0   
1        102      Bob    Neurology   Dr. John  6233.333333           15000.0   
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000               NaN   
3        104    David   Cardiology  Dr. Smith  6200.000000               NaN   
4        105      Eva  Dermatology   Dr. Rose  6233.333333               NaN   

   FinalAmount  
0       3000.0  
1       3000.0  
2          NaN  
3          NaN  
4          NaN  
